# Lesson 03: Dimensionality Reduction and Latent Variables - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: PCA, PLS, multicollinearity, and sensor fusion.

## Learning goals
1. Create a highly correlated sensor matrix.
2. Use PCA to identify latent dimensions.
3. Compare PCA regression with PLS regression.
4. Explain when variance is not the same thing as predictive relevance.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(3)
n = 120
latent_temp = rng.normal(0, 1, n)
latent_degradation = rng.normal(0, 0.4, n)
X = np.column_stack([
    latent_temp + rng.normal(0, 0.05, n),
    0.9*latent_temp + rng.normal(0, 0.05, n),
    -0.8*latent_temp + rng.normal(0, 0.05, n),
    latent_degradation + rng.normal(0, 0.08, n),
    0.5*rng.normal(size=n),
])
y = 2.5*latent_degradation + 0.3*latent_temp + rng.normal(0, 0.1, n)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=4)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA().fit(X_train_s)
plt.figure(figsize=(6, 3.5))
plt.bar(range(1, 6), pca.explained_variance_ratio_)
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.grid(axis="y", alpha=0.3)
plt.show()

pca2 = PCA(n_components=2).fit(X_train_s)
Z_train = pca2.transform(X_train_s)
Z_test = pca2.transform(X_test_s)
pc_reg = LinearRegression().fit(Z_train, y_train)
pls = PLSRegression(n_components=2).fit(X_train_s, y_train)

print("PCA+linear regression R2:", r2_score(y_test, pc_reg.predict(Z_test)))
print("PLS regression R2:", r2_score(y_test, pls.predict(X_test_s).ravel()))
print("\nInterpretation: PCA preserves high-variance directions. PLS searches for directions that also explain y.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?